In [30]:
import pandas as pd
import numpy as np
import hashlib
import re

In [19]:
MBIC_raw = pd.read_excel("~/BABE/Neural-Media-Bias-Detection-Using-Distant-Supervision-With-BABE/data/raw_labels_MBIC.xlsx")
SG1_raw = pd.read_excel("~/BABE/Neural-Media-Bias-Detection-Using-Distant-Supervision-With-BABE/data/raw_labels_SG1.xlsx")
SG2_raw = pd.read_excel("~/BABE/Neural-Media-Bias-Detection-Using-Distant-Supervision-With-BABE/data/raw_labels_SG2.xlsx")

In [3]:
# MBIC_raw.columns

In [4]:
SG1_raw.columns

Index(['text', 'news_link', 'outlet', 'topic', 'type', 'label_bias',
       'label_opinion', 'biased_words', 'Label_bias_0-1', 'annotator_id'],
      dtype='str')

In [20]:
babe_annotator_dems = pd.read_csv('~/BABE/Neural-Media-Bias-Detection-Using-Distant-Supervision-With-BABE/annotator_demographics.csv', delimiter = ';')

sg1_annos = set(SG1_raw['annotator_id'].to_list())
sg2_annos = set(SG2_raw['annotator_id'].to_list())
miss_annos = set(babe_annotator_dems['annotator_id'].to_list()) - set(SG1_raw['annotator_id'].to_list() + SG2_raw['annotator_id'].to_list()) # = set()
babe_annotator_dems.head()

,annotator_id,gender,English proficiency,age,education,study program,political orientation,news consumption,news outlets
0,4,male,Non-native,24,Bachelor's degree,MA inComputer Science,0,Every day,"Reuters,Japan Times,NHK World News,TVN24"
1,10,female,Near-native speaker,29,Bachelor's degree,MA in Intercultural Communication,-15,Several times per week,"Suedddeutsche Zeitung,Zeit,BBC News,France24"
2,9,male,Non-native,30,Bachelor's degree,MA in Social and Economic Data Science,-5,Several times per day,"Sueddeutsche Zeitung,CNN,NY Times"
3,2,male,Near-native speaker,25,Bachelor's degree,MA in Social and Economic Data Science,-15,Every day,"Sueddeutsche Zeitung,Zeit,Tagesspiegel"
4,3,female,Non-native,25,Bachelor's degree,MA in Computer Science,0,Several times per month,New York Times


In [31]:
FINAL_COLS = [
    "dataset",
    "survey_record_id", "sentence_id", "sentence_group_id", "created_at",
    "label_bias", "words", "label_opinion", "group_id", "text", "news_link",
    "type", "topic", "outlet", "mturk_id", "age", "gender", "education",
    "native_english_speaker", "political_ideology", "followed_news_outlets",
    "news_check_frequency", "survey_completed",
]

# Rename BABE annotator demographic columns into your desired schema
DEM_RENAME = {
    "annotator_id": "annotator_id",
    "gender": "gender",
    "English proficiency": "native_english_speaker",
    "age": "age",
    "education": "education",
    "political orientation": "political_ideology",
    "news consumption": "news_check_frequency",
    "news outlets": "followed_news_outlets",
}

def clean_cols(df):
    df = df.copy()
    df.columns = df.columns.str.strip()
    return df

def prep_sg(df, dataset_name):
    df = clean_cols(df)
    df["dataset"] = dataset_name

    # For SG1/SG2, annotator_id is the annotator identifier.
    # Since your target schema has mturk_id but not annotator_id, use annotator_id as mturk_id.
    if "annotator_id" in df.columns and "mturk_id" not in df.columns:
        df["mturk_id"] = df["annotator_id"]

    return df

# Prepare SG rows
sg1 = prep_sg(SG1_raw, "SG1")
sg2 = prep_sg(SG2_raw, "SG2")

# for prefix, frame in [('SG1', SG1_raw), ('SG2', SG2_raw)]:
#     frame['sentence_id_manual'] = [
#         f'{prefix}_{i}' for i in range(1, len(frame) + 1)
#     ]
#     if 'sentence_id' in frame.columns:
#         frame['sentence_id'] = frame['sentence_id'].fillna(frame['sentence_id_manual'])
#     else:
#         frame['sentence_id'] =frame['sentence_id_manual']


# def make_sentence_id(text, prefix):
#     if pd.isna(text):
#         return pd.NA

#     normalized = str(text).strip()
#     return f"{prefix}_{hashlib.md5(normalized.encode('utf-8')).hexdigest()}"


# for prefix, frame in [("SG1", SG1_raw), ("SG2", SG2_raw)]:
#     generated_ids = frame["text"].map(
#         lambda text: make_sentence_id(text, prefix)
#     )

#     if "sentence_id" not in frame.columns:
#         frame["sentence_id"] = generated_ids
#     else:
#         frame["sentence_id"] = frame["sentence_id"].fillna(generated_ids)




sg_all = pd.concat([sg1, sg2], ignore_index=True)


# #trying?
# sg_all['sentence_id'] = sg_all['sentence_id'].fillna(sg_all['sentence_id_manual']

# Prepare demographics
dems = clean_cols(babe_annotator_dems).rename(columns=DEM_RENAME)

# Keep only relevant demographic columns plus merge key
dem_cols = [
    "annotator_id",
    "age",
    "gender",
    "education",
    "native_english_speaker",
    "political_ideology",
    "followed_news_outlets",
    "news_check_frequency",
]
dem_cols = [c for c in dem_cols if c in dems.columns]

# Merge demographics onto SG rows
merged = sg_all.merge(
    dems[dem_cols],
    on="annotator_id",
    how="left",
    suffixes=("", "_dem"),
)

# If SG data already had some demographic cols, fill missing values from demographics merge
for col in [
    "age",
    "gender",
    "education",
    "native_english_speaker",
    "political_ideology",
    "followed_news_outlets",
    "news_check_frequency",
]:
    dem_col = f"{col}_dem"
    if dem_col in merged.columns:
        merged[col] = merged[col].combine_first(merged[dem_col])
        merged = merged.drop(columns=[dem_col])

# Add any missing final columns as blank
for col in FINAL_COLS:
    if col not in merged.columns:
        merged[col] = pd.NA
        
 

MBIC_raw['dataset'] = "MBIC"
# Reorder to final schema
sg1_sg2_final = merged[FINAL_COLS].copy()
adf = pd.concat([sg1_sg2_final, MBIC_raw]).copy()



# having too many issues with the sentence id mess, so just making all new ids - i think that makes sense:
def sid(s):
    s = str(s).strip().lower()
    s = re.sub(r'\s+', ' ', s)          # collapse internal whitespace/newlines
    s = s.replace('\u2019', "'").replace('\u201c','"').replace('\u201d','"')  # smart quotes
    return hashlib.md5(s.encode()).hexdigest()

adf['sentence_id'] = adf['text'].map(sid)

# sg1_sg2_final.head()

In [26]:
# adf.head()
SG1_raw.shape[0] + SG2_raw.shape[0] + MBIC_raw.shape[0] == adf.shape[0]

True

In [27]:
adf['sentence_id'].head()

0    SG1_72ddb14b3cc3d5295c150ed2b6dccba2
1    SG1_87ecd31bda2cfa11d2bcb6df3f59d55e
2    SG1_43359ef058448a46c6ccd6fe083bf948
3    SG1_0925a0ae5fb317fe0f78d7d0c6ddafa5
4    SG1_e3df3eac17628deff53d9429f4158eee
Name: sentence_id, dtype: str

In [32]:
adf.to_csv('full_babe_with_cats.csv')

In [23]:
# sg1_sg2_final['education'].value_counts()
# sg1_sg2_final.shape   31959


# sg1_sg2_final['sentence_id'].value_counts()
# sg1_sg2_final.columns
# SG1_raw.head()
adf.shape

(49734, 31)

In [24]:
adf['text'] = adf['text'].astype(str).fillna("")
adf["followed_news_outlets_clean"] = (
    adf["followed_news_outlets"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.replace("[", "", regex=False)
    .str.replace("]", "", regex=False)
    .str.replace("'", "", regex=False)
    .str.replace('"', "", regex=False)
)
adf["num_foll_news_outlet"] = adf["followed_news_outlets_clean"].apply(
    lambda x: len([item for item in x.split(",") if item.strip() != ""])
)

adf["num_age"] = pd.cut(adf["age"],bins=[-np.inf, 18, 30, 45, 55, 65, np.inf],labels=False)

adf["num_poli_idealogy"] = pd.cut(adf["political_ideology"],
                                 bins=[-np.inf, -6, -1, 1, 5, np.inf],
                                 labels=False)

adf["num_foll_news_outlet"] = pd.cut(adf["num_foll_news_outlet"],
                                          bins=[-np.inf, 1, 2, 3, 4, 6, np.inf],
                                          labels=False)


adf["num_news_check_frequency"] = adf["news_check_frequency"].map({
    "Never": 0,
    "Very rarely": 1,
    "Several times per month": 2,
    "Several times per week": 3,
    "Every day": 4,
    "Several times per day": 5,
})

def bin_education(edu):
    edu = str(edu).strip()
    if edu in {"Graduate work"}:
        return 5
    if edu in {"BA", "Bachelor", "Bachelors", "Bachelor's degree", "Bachelor’s degree"}:
        return 4
    if edu in {"Associate degree"}:
        return 3
    if edu in {"Some college", "Vocational or technical school"}:
        return 2
    if edu == "High school graduate":
        return 1
    return 0

adf["num_education"] = adf["education"].apply(bin_education)
adf['num_gender'] = adf['gender'].map({
    'Male': 1, 'male': 1,
    'Female': 0, 'female': 0,
}).fillna(2).astype(int)
adf["label_binary"] = adf["label_bias"].str.startswith("Biased").astype(int)


def binary_entropy(p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return -p * np.log(p) - (1 - p) * np.log(1 - p)

def h_cond_given_demo(group, demo_col, label_col='label_binary'):
    """H(Y | D=demo_col) for one sentence's ratings."""
    total = len(group)
    if total == 0:
        return 0.0
    h = 0.0
    for _, sub in group.groupby(demo_col):
        n = len(sub)
        weight = n / total
        p_biased = sub[label_col].mean()
        h += weight * binary_entropy(p_biased)
    return h

out_cat_names = ["num_age", "num_gender", "num_education", 
                 "num_poli_idealogy", "num_foll_news_outlet"]

rows = []
for sent_id, group in adf.groupby('sentence_id'):  # whatever your sentence-id column is called
    n_raters = len(group)
    p_biased = group['label_binary'].mean()
    h_obs = binary_entropy(p_biased)
    
    h_cond_by_demo = {demo: h_cond_given_demo(group, demo) for demo in out_cat_names}
    h_cond_avg = np.mean(list(h_cond_by_demo.values()))
    
    rows.append({
        'sentence_id': sent_id,
        'n_raters': n_raters,
        'p_biased': p_biased,
        'h_obs': h_obs,
        **{f'h_cond_{demo}': v for demo, v in h_cond_by_demo.items()},
        'h_cond_avg': h_cond_avg,
        'I_avg': h_obs - h_cond_avg,  # mutual information, averaged across demographics
    })

sentence_df = pd.DataFrame(rows)

In [27]:
# adf.columns
# sentence_df['I_avg'].describe()
sdf = sentence_df.merge(
    adf[['sentence_id', 'dataset']].drop_duplicates(),
    on='sentence_id',
    how='left'
)

sdf.shape

(33659, 12)

In [32]:
# sdf.shape
# sentence_df.shape
# adf['sentence_id'].nunique()
# sdf[sdf['dataset' == 'MBIC']]['I_avg'].describe()
# sdf['dataset'].value_counts()
# sentence_df['I_avg'].describe()
sentence_df['I_avg'].max()

np.float64(0.3681272785079508)